### `match` feature
One Python feature I still do not see many people use is the `match` statement.

It was added in Python 3.10, and for simple cases, you can think of it roughly like a switch statement from other languages.

For example, let's say you have a `status` value.
```py
match status:
    case "success":
        return "Operation completed"
    case "error":
        return "An error occurred"
    case "pending":
        return "Still in progress"
    case _:
        return f"Unknown: {status}"
```
Pretty simple.

If the status is `"success"`, it runs the success case.

If it is `"error"`, it runs the error case.

If nothing matches, it falls into the `_` case, which works like a default case.

So instead of writing a long chain of `if`, `elif`, and `else`, you can clean it up with `match`.

But that is not the most interesting part.

The real reason this feature is useful is that it does not only match simple values. It can also match the structure of the data.

For example:
```py
match data:
    case 0:
        return "Zero"

    case 1 | 2 | 3:
        return "Small number"

    case [first, second]:
        return f"Two items: {first} and {second}"

    case {"name": name, "age": age}:
        return f"Person: {name}, age {age}"

    case str(text):
        return f"String: {text}"

    case _:
        return f"Unknown: {data}"
```
This is where `match` becomes more powerful than a normal switch statement.

You can match the exact value `0`.

You can match multiple values with `1 | 2 | 3`.

You can match a list with two items, like `["apple", "banana"]`, and Python will pull out the first and second values for you.

You can match a dictionary that has `name` and `age` keys.

You can even match anything that is a string.

That is why this feature is called structural pattern matching.

With `match`, you are not only comparing values. You are describing the shape of data you want to handle, and Python can pull out the useful parts for you.

One important thing to remember is that Python checks the cases from top to bottom.

The first case that matches is the one that runs.

So the order matters.

Used in the right place, `match` can make your code much easier to read, especially when you are dealing with different types of data and you need to handle each one differently.

It is a small feature, but it is worth trying the next time your data-handling code starts getting messy.

---

# Structural Pattern Matching — `match / case` (Python 3.10+)

## What makes it different from switch / case in other languages

In Java or C, `switch` compares a single value against a list of constants — that is it.
Python's `match` can do that too, but the real feature is **structural pattern matching**:
inspecting the *shape* of the data (type, keys present, sequence length, object attributes)
and extracting parts of it into variables — all in one step.

One `match` block can replace a chain of `isinstance()` checks, key-existence tests,
`len()` guards, and variable assignments.

| Pattern type | Looks like | Matches | Extracts |
|---|---|---|---|
| Value | `"ok"`, `404` | exact equality | nothing |
| Sequence | `[a, b, c]` | list/tuple of length 3 | `a`, `b`, `c` |
| Mapping | `{"key": val}` | dict *containing* that key | `val` |
| Class | `Point(x=x, y=y)` | instance of `Point` | `x`, `y` |


### 1. Value matching + OR patterns — the "boring" baseline that still beats a long `if`/`elif` chain

In [2]:
def http_label(status: int) -> str:
    match status:
        case 200 | 201 | 204:
            return "success"
        case 301 | 302:
            return "redirect"
        case 400:
            return "bad request"
        case 401 | 403:
            return "auth error"
        case 404:
            return "not found"
        case 500 | 502 | 503:
            return "server error"
        case _:
            return f"unknown ({status})"


for s in [200, 201, 301, 400, 401, 404, 500, 999]:
    print(f"  {s}  ->  {http_label(s)}")


  200  ->  success
  201  ->  success
  301  ->  redirect
  400  ->  bad request
  401  ->  auth error
  404  ->  not found
  500  ->  server error
  999  ->  unknown (999)


### 2. Sequence patterns — matching by length and structure

A sequence pattern matches a list or tuple by its shape: how many elements it has,
and what those elements look like — without any `len()` calls or index access.

```python
match command:
    case []:                    # exactly empty
    case [action]:              # exactly one element, captured as action
    case [action, target]:      # exactly two elements
    case [action, *rest]:       # one + any number of extras (rest is a list)
    case [first, *_, last]:     # first and last captured, middle discarded
```

The `*` splice works like `*args` — "zero or more of whatever is left".
A guard (`if` clause after the pattern) adds extra conditions on the captured values.


In [4]:
a, *_, b = ('a', 1, 2, 3, 'b')

In [5]:
a

'a'

In [6]:
b

'b'

In [7]:
_

[1, 2, 3]

```py
async def _build_reaction_summary(
        self, user_id: int, movie_id: int
    ) -> MovieReactionSummarySchema:
    ...
    if user_reaction is True:
        reaction_str = "like"
    elif user_reaction is False:
        reaction_str = "dislike"
    else:
        reaction_str = None
```
This can be refactored to:
```py
async def _build_reaction_summary(
        self, user_id: int, movie_id: int
    ) -> MovieReactionSummarySchema:
    ...
    match user_reaction:
        case True:
            reaction_str = "like"
        case False:
            reaction_str = "dislike"
        case _:
            reaction_str = None
```
`_` is the "catch-all" wildcard in `match`, whereas `case None:` only matches the literal `None` value. Both work identically given the actual data, but `_` signals "anything else" more clearly to the reader.

In [1]:
# Sequence pattern — a CLI-style command dispatcher
def dispatch(command: list[str]) -> str:
    match command:
        case []:
            return "empty command"

        case ["help"]:
            return "Available: help, create, delete, list, search"

        case ["create", resource]:
            return f"Creating {resource}"

        case ["delete", resource, "--force"]:
            return f"Force-deleting {resource} — skipping confirmation"

        case ["delete", resource]:
            return f"Deleting {resource} — will ask for confirmation"

        # guard: *filters must be non-empty
        case ["list", *filters] if filters:
            return f"Listing with filters: {filters}"

        case ["list"]:
            return "Listing all resources"

        case ["search", query, *options]:
            return f"Searching '{query}' — options: {options or 'none'}"

        case [unknown, *_]:
            return f"Unknown command: {unknown!r}"

In [ ]:
commands = [
    [],
    ["help"],
    ["create", "user"],
    ["delete", "user:42"],
    ["delete", "user:42", "--force"],
    ["list"],
    ["list", "--active", "--role=admin"],
    ["search", "alice"],
    ["search", "alice", "--limit=10", "--sort=asc"],
    ["fly", "to", "the", "moon"],
]

for cmd in commands:
    print(f"  {str(cmd):<45} -> {dispatch(cmd)}")

## Mapping patterns — matching dict keys and extracting values

A mapping pattern matches any dict that *contains at least* the specified keys with
the specified values. **Other keys are ignored** — this is a partial match, not equality.

```python
match response:
    case {"status": "ok", "data": data}:   # must have both keys; data value is captured
        ...
    case {"status": "error"}:              # any dict with status == "error"; extras ignored
        ...
```

> **The key insight:** `case {"status": "ok"}:` matches `{"status": "ok", "extra": 42}`.
> But `d == {"status": "ok"}` returns `False` for that same dict.
> Pattern matching asks "does this shape exist inside the data?" — equality asks "is it exactly this?"

Patterns are tested **top to bottom** — put more specific patterns (more keys, nested structure)
before less specific ones. First match wins; there is no fall-through.

Use `**rest` to capture the remaining keys if you need them:
```python
case {"type": "login", **rest}:   # rest = all other key-value pairs
    ...
```


In [3]:
# Mapping pattern — dispatching on different API response shapes
def handle_response(response: dict) -> str:
    match response:
        # Nested mapping: "data" must itself be a dict containing a "users" list
        case {"status": "ok", "data": {"users": users}} if isinstance(users, list):
            return f"Fetched {len(users)} users"

        case {"status": "ok", "data": data}:
            keys = list(data) if isinstance(data, dict) else repr(data)
            return f"OK — payload: {keys}"

        # Most specific error first: has both "code" and "message"
        case {"status": "error", "code": code, "message": msg}:
            return f"Error {code}: {msg}"

        case {"status": "error", "message": msg}:
            return f"Error (no code): {msg}"

        case {"status": "error"}:
            return "Error (no details provided)"

        # Has a "status" key but an unexpected value
        case {"status": unknown_status}:
            return f"Unexpected status: {unknown_status!r}"

        # Missing "status" entirely
        case _:
            return "Malformed response — missing 'status' key"


responses = [
    {"status": "ok",    "data": {"users": [1, 2, 3]}},
    {"status": "ok",    "data": {"order_id": 99, "total": 149.99}},
    {"status": "error", "code": 429, "message": "rate limited"},
    {"status": "error", "message": "db connection timeout"},
    {"status": "error"},
    {"status": "deprecated", "data": {}},
    {"result": "ok"},
]

for resp in responses:
    print(f"  {str(resp):<55} -> {handle_response(resp)}")


  {'status': 'ok', 'data': {'users': [1, 2, 3]}}          -> Fetched 3 users
  {'status': 'ok', 'data': {'order_id': 99, 'total': 149.99}} -> OK — payload: ['order_id', 'total']
  {'status': 'error', 'code': 429, 'message': 'rate limited'} -> Error 429: rate limited
  {'status': 'error', 'message': 'db connection timeout'} -> Error (no code): db connection timeout
  {'status': 'error'}                                     -> Error (no details provided)
  {'status': 'deprecated', 'data': {}}                    -> Unexpected status: 'deprecated'
  {'result': 'ok'}                                        -> Malformed response — missing 'status' key


## Class patterns — match by type AND attributes simultaneously

A class pattern checks `isinstance(event, SomeClass)` AND extracts named attributes
into variables — in one step.

**Without match:**
```python
if isinstance(event, OrderPlaced):
    if event.amount > 1000:
        flag_for_review(event.order_id)
    else:
        process_normally(event.order_id, event.user_id)
elif isinstance(event, PaymentFailed):
    alert_team(event.order_id, event.reason)
```

**With match:**
```python
match event:
    case OrderPlaced(order_id=oid, amount=amount) if amount > 1000:
        flag_for_review(oid)
    case OrderPlaced(order_id=oid, user_id=uid, amount=amount):
        process_normally(oid, uid, amount)
    case PaymentFailed(order_id=oid, reason=reason):
        alert_team(oid, reason)
```

**Works automatically with `@dataclass`** — they generate `__match_args__`,
which also enables positional patterns `Point(x, y)` without keyword names.
For plain classes, always use keyword patterns `ClassName(attr=var)` — always safe.


In [4]:
from dataclasses import dataclass


@dataclass
class UserCreated:
    user_id: int
    email:   str

@dataclass
class OrderPlaced:
    order_id: int
    user_id:  int
    amount:   float

@dataclass
class PaymentFailed:
    order_id: int
    reason:   str

@dataclass
class ItemOutOfStock:
    product_id:         int
    quantity_requested: int


def handle_event(event: object) -> str:
    match event:
        case UserCreated(user_id=uid, email=email):
            return f"[EMAIL]   welcome -> {email} (user #{uid})"

        # Guard on the same type: high-value orders take a different path
        case OrderPlaced(order_id=oid, amount=amount) if amount > 1_000:
            return f"[REVIEW]  high-value order #{oid}: ${amount:.2f}"

        case OrderPlaced(order_id=oid, user_id=uid, amount=amount):
            return f"[PROCESS] order #{oid} for user #{uid}: ${amount:.2f}"

        case PaymentFailed(order_id=oid, reason=reason):
            return f"[ALERT]   payment failed for order #{oid} -- {reason}"

        case ItemOutOfStock(product_id=pid, quantity_requested=qty):
            return f"[NOTIFY]  product #{pid} out of stock (requested {qty})"

        case _:
            return f"[SKIP]    unhandled: {type(event).__name__}"


events: list[object] = [
    UserCreated(user_id=1, email="alice@example.com"),
    OrderPlaced(order_id=101, user_id=1, amount=49.99),
    OrderPlaced(order_id=102, user_id=2, amount=1_500.00),
    PaymentFailed(order_id=101, reason="card declined"),
    ItemOutOfStock(product_id=55, quantity_requested=3),
]

for event in events:
    print(handle_event(event))


[EMAIL]   welcome -> alice@example.com (user #1)
[PROCESS] order #101 for user #1: $49.99
[REVIEW]  high-value order #102: $1500.00
[ALERT]   payment failed for order #101 -- card declined
[NOTIFY]  product #55 out of stock (requested 3)


## Gotchas

### 1. Mapping pattern is a partial match — not equality

```python
data = {"status": "ok", "extra": 42}

match data:
    case {"status": "ok"}:   # MATCHES — extra keys are ignored
        ...

data == {"status": "ok"}     # False — equality requires an exact match
```

### 2. `_` does not bind — a bare name does

```python
case {"id": _}:    # has "id" key — value is discarded, not bound
case {"id": uid}:  # has "id" key — value captured into uid
case _:            # wildcard — matches anything, binds nothing
case x:            # also matches anything — binds the whole value to x
```

### 3. Sequence patterns match both lists AND tuples

```python
match (1, 2, 3):
    case [a, b, c]:   # MATCHES — tuple satisfies a sequence pattern
        ...
```

If you need to enforce the type, add a guard:
```python
case [a, b, c] if isinstance(container, tuple):
    ...
```

### 4. `match` is a soft keyword — not a reserved word

`match` can still be used as a variable name anywhere — no breaking change for existing code:
```python
match = re.compile(r"\d+")   # valid in Python 3.10+
match status:                  # also valid — context resolves the ambiguity
    ...
```

---

## Interview angle

The question *"what is structural pattern matching?"* is testing whether you know it goes
beyond value comparison. The key things to mention:
- matching by shape and extracting values in the same step
- the partial-match behavior of mapping patterns

The follow-up is almost always:
**"How is `case {"status": "ok"}:` different from `if d == {"status": "ok"}:`?"**
The pattern matches any dict that *contains* that key-value pair, regardless of other keys.
Equality requires the dict to be exactly `{"status": "ok"}` and nothing else.


In [5]:
# Gotcha 1: mapping pattern is a PARTIAL match, not equality
data = {"status": "ok", "extra": 42, "timestamp": "2026-01-01"}

match data:
    case {"status": "ok"}:
        print("pattern matched -- extra keys are irrelevant to the mapping pattern")
    case _:
        print("no match")

pattern = {"status": "ok"}
print(f"direct equality  data == {pattern!r}:  {data == pattern}")
print()

# Gotcha 2: _ (wildcard) does not bind; a bare name does
def demo_capture(val: dict) -> None:
    match val:
        case {"id": _}:       # "id" value is NOT captured
            print(f"  has 'id' key -- value discarded by _")
        case {"name": name}:  # "name" value IS captured
            print(f"  has 'name' key -- captured: {name!r}")
        case _:
            print("  no match")

demo_capture({"id": 99,  "role": "admin"})
demo_capture({"name": "Alice", "role": "user"})
print()

# Gotcha 3: sequence patterns match both lists AND tuples
for container in ([1, 2, 3], (1, 2, 3)):
    match container:
        case [a, b, c]:
            print(f"  {type(container).__name__:<6}  [a, b, c] matched -> a={a}, b={b}, c={c}")


pattern matched -- extra keys are irrelevant to the mapping pattern
direct equality  data == {'status': 'ok'}:  False

  has 'id' key -- value discarded by _
  has 'name' key -- captured: 'Alice'

  list    [a, b, c] matched -> a=1, b=2, c=3
  tuple   [a, b, c] matched -> a=1, b=2, c=3


|Cell|Pattern type|What you see|
|---|---|---|
|sm001|—|Intro table: 4 pattern types|
|sm002|Value + OR (`\|`)|HTTP status codes — replaces long if/elif|
|sm003|—|Sequence pattern syntax|
|sm004|Sequence|CLI command dispatcher — `*rest`, guards, `--force` flag|
|sm005|—|Mapping pattern — the partial-match insight|
|sm006|Mapping (nested)|API response dispatcher — nested dict pattern, most-specific-first ordering|
|sm007|—|Class pattern explanation vs. isinstance chains|
|sm008|Class + guard|Event dispatcher — `UserCreated`, `OrderPlaced` (with high-value guard), `PaymentFailed`, `ItemOutOfStock`|
|sm009|—|Gotchas + interview angle|
|sm010|—|Live demos: partial match proof, `_` vs capture, list/tuple symmetry|

The two things worth owning before an interview: sequence patterns work on both lists and tuples (no type distinction), and mapping patterns are partial matches — `case {"status": "ok"}:` matches any dict that _contains_ that key, not just a dict with that single key.